# 🚗 Car Price Prediction — Random Forest Regressor
**Author:** Mr. Vutha sokban 

**Algorithm:** Random Forest Regressor with GridSearchCV Hyperparameter Tuning  
**Dataset:** [Vehicle Dataset from CarDekho — Kaggle](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho) now make sildes of that topic which involved with model compare(random forest and liner regression) and the accuracy, result, whi h is the best and conclusion and as such) anything that lacking you can add at maximun is 11 slides

## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ All libraries imported successfully!')

## 📂 2. Load Dataset

> **Instructions:** Download `car data.csv` from [Kaggle](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho) and place it in the same folder as this notebook.

In [ ]:
# Load the dataset
df = pd.read_csv(r'C:\Users\ASUS\Downloads\car data.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
df.info()

In [ ]:
# Statistical summary
print('=== Statistical Summary ===')
df.describe()

In [ ]:
# Check missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found ✅')

In [ ]:
# Check duplicate rows
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')

In [ ]:
# Distribution of target variable (Selling Price)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Selling_Price'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Selling Price', fontsize=14)
axes[0].set_xlabel('Selling Price (in Lakhs)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['Selling_Price']), bins=30, color='coral', edgecolor='white')
axes[1].set_title('Log Distribution of Selling Price', fontsize=14)
axes[1].set_xlabel('Log(Selling Price)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('eda_selling_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# Categorical columns distribution
cat_cols = ['Fuel_Type', 'Seller_Type', 'Transmission']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color=sns.color_palette('muted', len(counts)))
    axes[i].set_title(f'{col} Distribution', fontsize=13)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# Selling Price vs Present Price scatter
plt.figure(figsize=(8, 5))
plt.scatter(df['Present_Price'], df['Selling_Price'], alpha=0.5, color='teal')
plt.title('Present Price vs Selling Price', fontsize=14)
plt.xlabel('Present Price (in Lakhs)')
plt.ylabel('Selling Price (in Lakhs)')
plt.tight_layout()
plt.savefig('eda_price_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# Average selling price by Fuel Type and Transmission
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('Fuel_Type')['Selling_Price'].mean().sort_values().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Avg Selling Price by Fuel Type', fontsize=13)
axes[0].set_xlabel('Fuel Type')
axes[0].set_ylabel('Avg Selling Price (Lakhs)')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('Transmission')['Selling_Price'].mean().sort_values().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Avg Selling Price by Transmission', fontsize=13)
axes[1].set_xlabel('Transmission Type')
axes[1].set_ylabel('Avg Selling Price (Lakhs)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_price_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

## 🛠️ 4. Data Preprocessing

In [ ]:
# Make a copy to preserve original
data = df.copy()

# 4.1 Feature Engineering: Car Age from Year
data['Car_Age'] = 2025 - data['Year']
print('✅ Created Car_Age feature')

# 4.2 Drop columns not needed for modeling
data.drop(columns=['Car_Name', 'Year'], inplace=True)
print('✅ Dropped Car_Name and Year columns')

# 4.3 Encode categorical variables
le = LabelEncoder()
for col in ['Fuel_Type', 'Seller_Type', 'Transmission']:
    data[col] = le.fit_transform(data[col])
    print(f'✅ Encoded: {col}')

print(f'\nFinal dataset shape: {data.shape}')
data.head()

In [ ]:
# Correlation heatmap (after encoding)
plt.figure(figsize=(10, 7))
corr = data.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# 4.4 Define Features (X) and Target (y)
X = data.drop(columns=['Selling_Price'])
y = data['Selling_Price']

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'\nFeature columns: {list(X.columns)}')

In [ ]:
# 4.5 Train / Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set:  {X_train.shape[0]} samples')
print(f'Test set:      {X_test.shape[0]} samples')

## 🌲 5. Random Forest — Baseline Model

In [ ]:
# Train baseline Random Forest
rf_base = RandomForestRegressor(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)

# Predict
y_pred_base = rf_base.predict(X_test)

# Metrics
mae_base  = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base   = r2_score(y_test, y_pred_base)

print('=== Baseline Random Forest ===')
print(f'  MAE  : {mae_base:.4f}')
print(f'  RMSE : {rmse_base:.4f}')
print(f'  R²   : {r2_base:.4f}')

## ⚙️ 6. Hyperparameter Tuning — GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

print(f'Total parameter combinations: {3*3*2*2}')
print('Running GridSearchCV with 5-fold CV... (this may take a minute)')

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\n✅ Best Parameters: {grid_search.best_params_}')
print(f'   Best CV R²:      {grid_search.best_score_:.4f}')

## 🏆 7. Best Model — Evaluation on Test Set

In [ ]:
# Evaluate best model
best_rf = grid_search.best_estimator_
y_pred  = best_rf.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

# Cross-validation score
cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='r2')

print('=======================================')
print('  Tuned Random Forest — Test Results   ')
print('=======================================')
print(f'  MAE              : {mae:.4f}')
print(f'  RMSE             : {rmse:.4f}')
print(f'  R² (test)        : {r2:.4f}')
print(f'  R² (5-fold CV)   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=======================================')

In [ ]:
# Before vs After Tuning comparison table
comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Baseline RF': [round(mae_base, 4), round(rmse_base, 4), round(r2_base, 4)],
    'Tuned RF':    [round(mae, 4),      round(rmse, 4),      round(r2, 4)]
})
print('=== Baseline vs Tuned Random Forest ===')
comparison

## 📊 8. Visualizations

In [ ]:
# 8.1 Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=60)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Selling Price (Lakhs)', fontsize=12)
plt.ylabel('Predicted Selling Price (Lakhs)', fontsize=12)
plt.title(f'Actual vs Predicted — Random Forest (R² = {r2:.4f})', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig('rf_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# 8.2 Residual Plot
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.5, color='coral', edgecolors='white', s=60)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Values', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].set_title('Residuals vs Predicted', fontsize=13)

# Residual distribution
axes[1].hist(residuals, bins=30, color='teal', edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual Value', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=13)

plt.suptitle('Random Forest — Residual Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# 8.3 Feature Importance
feature_importance = pd.Series(
    best_rf.feature_importances_, index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors = sns.color_palette('Blues_d', len(feature_importance))
feature_importance.plot(kind='barh', color=colors)
plt.xlabel('Feature Importance Score', fontsize=12)
plt.title('Random Forest — Feature Importances', fontsize=14)
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# 8.4 Cross-Validation Score Distribution
plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='white')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean R² = {cv_scores.mean():.4f}')
plt.xlabel('Fold', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('5-Fold Cross-Validation R² Scores — Random Forest', fontsize=13)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig('rf_cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

## 📋 9. Summary

This notebook implements a **Random Forest Regressor** for predicting used car selling prices.

In [ ]:
# Final Summary Table
summary = pd.DataFrame({
    'Model':      ['Baseline RF (100 trees)', f'Tuned RF {grid_search.best_params_}'],
    'MAE':        [round(mae_base, 4), round(mae, 4)],
    'RMSE':       [round(rmse_base, 4), round(rmse, 4)],
    'R² (test)':  [round(r2_base, 4), round(r2, 4)],
    'R² (CV)':    ['—', f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}"]
})

print('===========================================')
print('   RANDOM FOREST — FINAL SUMMARY TABLE    ')
print('===========================================')
summary

### Key Findings

- **Present Price** and **Car Age** are the most important features in predicting selling price.
- Hyperparameter tuning via GridSearchCV improved the R² score over the baseline.
- The residuals are approximately centered around zero, indicating a well-fitted model.
- Cross-validation confirms the model generalizes consistently across folds.

> 📌 **Next step:** Compare these results with the Linear Regression model (implemented by Mr. Vutha Sokban) in the comparative analysis section of the final presentation.